# Notebook 1: Data Engineering & ETL Pipeline

## 1. Business Context & Objective
ITB is a commercial enterprise distributing three main product lines (Product A, Product B, and the core cash-cow Product C) across four sales channels: Ecommerce, Website, Store (Agent), and Premium. 

Historically, data was collected from disparate transactional systems and stored in fragmented formats (including partitioned transaction tables and inconsistent customer master sheets). This led to critical data quality issues:
* Duplicate customer profiles with conflicting demographic and educational attributes.
* Inconsistent store channel classifications ("stores/premium" vs. distinct store/premium splits).
* Orphan transactional records missing valid customer keys.

**Objective of this Notebook:**
Establish a robust, reproducible, and production-grade Extract-Transform-Load (ETL) pipeline. We will ingest the raw transactional and master data, resolve systematic data anomalies, handle missing values scientifically, and export a clean, integrated master dataset (`master_df.csv`) to serve as the single source of truth for downstream diagnostic analytics and dimensional data warehousing.

In [16]:
import pandas as pd
import numpy as np
import os

# Configure pandas options for clean notebook output
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Environment successfully initialized with pandas and numpy.")

Environment successfully initialized with pandas and numpy.


## 2. Data Extraction & Consolidation

### 2.1. Ingestion Strategy
The raw dataset is divided into several files:
* `dim_customer.csv`: Customer master data containing demographic, education, and occupational details.
* `dim_product.csv`: Product catalog containing product codes and hierarchical categories.
* `dim_stores.csv`: Store master data containing geographical locations and selling channels.
* `dim_order_part_1.csv` through `dim_order_part_5.csv`: Transactional records partitioned into 5 parts due to file size constraints.

We will dynamically locate these partitioned transaction files, consolidate them into a unified `orders` DataFrame, and load the respective master dimensions.

In [17]:
# Robust path resolution to handle both root execution and notebook-folder execution
import os

# Path option 1: If executed from the project root directory
path_option_root = 'data/raw/'
# Path option 2: If executed from inside the 'notebooks' directory (go up one level)
path_option_notebook = '../data/raw/'

if os.path.exists(path_option_root):
    raw_data_path = path_option_root
elif os.path.exists(path_option_notebook):
    raw_data_path = path_option_notebook
else:
    raw_data_path = './'

print(f"Resolved raw data directory to: '{raw_data_path}'")

# Load Master Dimension Files
try:
    df_customer_raw = pd.read_csv(os.path.join(raw_data_path, 'dim_customer.csv'))
    df_stores_raw = pd.read_csv(os.path.join(raw_data_path, 'dim_stores.csv'))
    df_product_raw = pd.read_csv(os.path.join(raw_data_path, 'dim_product.csv'))
    print("Successfully loaded Master Dimension tables (Customer, Stores, Product).")
except FileNotFoundError as e:
    print(f"Error loading dimensions: {e}. Please check your folder structure.")

# Dynamically locate and concatenate partitioned Order tables
try:
    order_files = sorted([f for f in os.listdir(raw_data_path) 
                          if f.startswith('dim_order_part_') and f.endswith('.csv')])
    
    order_parts = []
    for file in order_files:
        part_path = os.path.join(raw_data_path, file)
        df_part = pd.read_csv(part_path)
        order_parts.append(df_part)
        print(f"Ingested: {file} | Shape: {df_part.shape}")
        
    df_order_raw = pd.concat(order_parts, ignore_index=True)
    print(f"\nSuccessfully consolidated transactional database. Combined shape: {df_order_raw.shape}")
except Exception as e:
    print(f"Error during transactional data consolidation: {e}")

Resolved raw data directory to: '../data/raw/'
Successfully loaded Master Dimension tables (Customer, Stores, Product).
Ingested: dim_order_part_1.csv | Shape: (500000, 12)
Ingested: dim_order_part_2.csv | Shape: (500000, 12)
Ingested: dim_order_part_3.csv | Shape: (500000, 12)
Ingested: dim_order_part_4.csv | Shape: (500000, 12)
Ingested: dim_order_part_5.csv | Shape: (107643, 12)

Successfully consolidated transactional database. Combined shape: (2107643, 12)


## 3. Initial Data Profiling & Structural Validation

Before applying transformations, we conduct initial data profiling to check the structure, check for nulls, assess data types, and identify structural issues across the raw DataFrames. 
This step helps us establish baseline metrics (such as row counts and data types) to verify data integrity post-transformation.

In [18]:
# Profile raw tables to evaluate structural health
datasets = {
    "Raw Customers": df_customer_raw,
    "Consolidated Orders": df_order_raw,
    "Raw Stores": df_stores_raw,
    "Raw Products": df_product_raw
}

for name, df in datasets.items():
    print(f"=== Profiling: {name} ===")
    print(f"Shape: {df.shape}")
    print("\nData Types & Non-Null Counts:")
    print(df.info())
    print("\nMissing Values Count:")
    print(df.isnull().sum())
    print("\nUnique Cardinality:")
    for col in df.columns:
        print(f" - {col}: {df[col].nunique()} unique values")
    print("="*50 + "\n")

=== Profiling: Raw Customers ===
Shape: (849472, 10)

Data Types & Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 849472 entries, 0 to 849471
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   customer_key     849472 non-null  int64 
 1   customer_id      849472 non-null  int64 
 2   name_full        849472 non-null  object
 3   gender           849472 non-null  object
 4   address          445211 non-null  object
 5   region           849472 non-null  object
 6   dob              849472 non-null  object
 7   age              849472 non-null  int64 
 8   education_level  685804 non-null  object
 9   job_title        670136 non-null  object
dtypes: int64(3), object(7)
memory usage: 64.8+ MB
None

Missing Values Count:
customer_key            0
customer_id             0
name_full               0
gender                  0
address            404261
region                  0
dob               

## 4. Data Cleaning, Type Standardization & Schema Alignment

In this phase, we align schemas and standardize data types across all tables:
1. Parse date columns (`order_date` and `dob`) into proper `datetime64[ns]` formats.
2. Standardize key column names (renaming `store_codes` to `store_code` in the Store master).
3. Handle missing crucial keys by removing null rows in the Store dimension.

In [19]:
print("--- Standardizing Date Types & Column Names ---")

# 1. Standardize date formats
df_order_raw['order_date'] = pd.to_datetime(df_order_raw['order_date'], errors='coerce')
df_customer_raw['dob'] = pd.to_datetime(df_customer_raw['dob'], errors='coerce')

# 2. Align store column name in dimension table
df_stores_raw.rename(columns={'store_codes': 'store_code'}, inplace=True)

# 3. Clean null store codes in the store dimension
initial_store_count = len(df_stores_raw)
df_stores_raw.dropna(subset=['store_code'], inplace=True)
df_stores_raw['store_code'] = df_stores_raw['store_code'].astype(int)

print(f"Standardized date formats for transactional and customer tables.")
print(f"Renamed 'store_codes' to 'store_code' in stores master.")
print(f"Removed {initial_store_count - len(df_stores_raw)} store records with null codes.")

--- Standardizing Date Types & Column Names ---
Standardized date formats for transactional and customer tables.
Renamed 'store_codes' to 'store_code' in stores master.
Removed 3 store records with null codes.


## 5. Resolving Channel Inconsistencies & Orphan Records

### 5.1. Channel Inconsistency Resolution
The store master table categorizes channels as `stores/premium`, while the order transaction table splits them into distinct `stores` and `premium` records. 
To resolve this without losing granularity, we analyze transaction behaviors:
* Stores with transactions in both `stores` and `premium` types within the Orders table will be classified as `premium` segment stores.
* The remaining stores in the `stores/premium` group will be standardized as `stores`.

### 5.2. Preserving Referential Integrity
We identify and remove transactional records containing `customer_id`s that do not exist in the customer dimension table (such as the dummy `customer_id = -43`), ensuring perfect relational integrity before warehousing.b

In [20]:
print("--- Resolving Store Channel Inconsistencies ---")

# Find store codes in orders that have transactions under both 'stores' and 'premium'
store_channel_counts = df_order_raw.groupby('store_code')['sales_channel'].nunique()
hybrid_stores = store_channel_counts[store_channel_counts > 1].index

# Update those store channels to 'premium' in store master, and the rest of 'stores/premium' to 'stores'
df_stores_raw.loc[df_stores_raw['store_code'].isin(hybrid_stores), 'sales_channel'] = 'premium'
df_stores_raw['sales_channel'].replace({'stores/premium': 'stores'}, inplace=True)

print(f"Successfully standardized hybrid channels. Unique channels left in Store master: {df_stores_raw['sales_channel'].unique()}")


print("\n--- Identifying and Removing Orphan Records ---")

# Find customer IDs in orders that do not exist in customer master
valid_customer_ids = set(df_customer_raw['customer_id'].unique())
order_customer_ids = set(df_order_raw['customer_id'].unique())
orphan_customer_ids = order_customer_ids - valid_customer_ids

print(f"Number of unique orphan Customer IDs found in Orders: {len(orphan_customer_ids)}")

# Filter out transactions belonging to orphan customer IDs (including ID -43)
initial_order_count = len(df_order_raw)
df_order_cleaned = df_order_raw[~df_order_raw['customer_id'].isin(orphan_customer_ids)].copy()
removed_orders = initial_order_count - len(df_order_cleaned)

print(f"Removed {removed_orders:,} orphan transaction records from consolidated orders.")
print(f"Verified referential integrity. New transactional count: {len(df_order_cleaned):,}")

--- Resolving Store Channel Inconsistencies ---


C:\Users\KIEU VI\AppData\Local\Temp\ipykernel_26124\239647754.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_stores_raw['sales_channel'].replace({'stores/premium': 'stores'}, inplace=True)


Successfully standardized hybrid channels. Unique channels left in Store master: ['premium' 'stores' 'ecommerce' 'website']

--- Identifying and Removing Orphan Records ---
Number of unique orphan Customer IDs found in Orders: 619
Removed 56,435 orphan transaction records from consolidated orders.
Verified referential integrity. New transactional count: 2,051,208


## 6. Advanced Attribute Standardization on Dimensions

To ensure high-quality segmentation and profiling, we perform advanced cleaning on customer attributes:
1. Standardize redundant educational categories (e.g., merging "Bachelor's degree" into "Bachelor", "High school education" into "High school").
2. Fill remaining missing values in categorical fields (`education_level`, `job_title`, `address`, `district`) with an explicit `'Unknown'` placeholder, which is standard data warehousing practice.
3. Apply string trimming and title casing to geographical variables to eliminate spelling duplicates caused by trailing spaces or case differences.

In [21]:
print("--- Standardizing Categorical Attributes ---")

# 1. Map and consolidate redundant education levels
education_mapping = {
    "Bachelor's degree": "Bachelor",
    "Master's degree": "Master",
    "High school education": "High school"
}
df_customer_raw['education_level'] = df_customer_raw['education_level'].replace(education_mapping)

# 2. Impute missing values with 'Unknown' (excluding non-existent 'district' column)
categorical_cols_to_fill = ['education_level', 'job_title', 'address']
for col in categorical_cols_to_fill:
    df_customer_raw[col] = df_customer_raw[col].fillna('Unknown')

# 3. Clean geographical strings in customer and store files
df_customer_raw['region'] = df_customer_raw['region'].str.strip().str.title()
df_stores_raw['name_region'] = df_stores_raw['name_region'].str.strip().str.title()
df_stores_raw['name_district'] = df_stores_raw['name_district'].str.strip().str.title()

# Fix the pandas 3.0 warning by avoiding inplace=True
df_stores_raw['sales_channel'] = df_stores_raw['sales_channel'].replace({'stores/premium': 'stores'})

print("Education categories mapped and missing demographics filled with 'Unknown'.")
print("Geographical text columns standardized with title casing.")

--- Standardizing Categorical Attributes ---
Education categories mapped and missing demographics filled with 'Unknown'.
Geographical text columns standardized with title casing.


## 7. Consolidation into the Single Source of Truth (master_df)

In this final phase of our ETL pipeline, we merge the cleaned tables and apply a critical temporal filter:

### 7.1. Filtering Incomplete Historical Data (Excluding Year 2021)
During initial profiling, we discovered that the year 2021 contains an extreme data sparsity issue, recording only a single day of transaction history. 
* **Analytical Impact:** Leaving these records in the dataset would introduce immense statistical bias. It would distort year-over-year (YoY) growth rate calculations (creating an artificial "hyper-growth" spike from 2021 to 2022) and skew diagnostic trend lines.
* **Engineering Decision:** To protect the integrity and consistency of our downstream diagnostic models, we exclude all transactions from the year 2021. Our analysis will strictly focus on the continuous, fully-recorded historical window of **2022, 2023, and 2024**.

### 7.2. Merging & Consolidation Logic
1. Merge the cleaned transactional records (`df_order_cleaned`) with the customer master (`df_customer_raw`) using a Left Join on `customer_id`.
2. Extract the transaction year (`year`) and filter out 2021 records.
3. Map physical store locations with store dimensions (`store_region` and `store_district`) by merging with unique store records on `store_code`.
4. Export the final consolidated dataset as `master_df.csv` to the `data/processed/` directory.

In [24]:
print("--- Merging Cleaned Datasets & Filtering Year 2021 ---")

# 1. Left join transactions with customer details
df_master = pd.merge(df_order_cleaned, df_customer_raw, on='customer_id', how='left')

# 2. Extract year from order_date and exclude year 2021 (due to incomplete single-day records)
df_master['year'] = df_master['order_date'].dt.year
initial_master_rows = len(df_master)

# Filter out year 2021
df_master = df_master[df_master['year'] != 2021].copy()
removed_2021_rows = initial_master_rows - len(df_master)

print(f"Successfully excluded {removed_2021_rows:,} transaction records from the year 2021.")

# 3. Get unique store locations (first record for each store_code) to prevent row-multiplication
store_locations = df_stores_raw[['store_code', 'name_region', 'name_district']].drop_duplicates(subset=['store_code'], keep='first')
store_locations.rename(columns={'name_region': 'store_region', 'name_district': 'store_district'}, inplace=True)

# 4. Merge store locations into master dataframe
df_master = pd.merge(df_master, store_locations, on='store_code', how='left')

print(f"Final merged dataset row count (excluding 2021): {len(df_master):,}")

# 5. Set up paths and export to CSV
parent_dir = os.path.dirname(os.path.normpath(raw_data_path)) 
processed_dir = os.path.join(parent_dir, 'processed')
os.makedirs(processed_dir, exist_ok=True)

master_file_path = os.path.join(processed_dir, 'master_df.csv')
df_master.to_csv(master_file_path, index=False, encoding='utf-8-sig')

print(f"Consolidated Single Source of Truth successfully exported to: '{master_file_path}'")

--- Merging Cleaned Datasets & Filtering Year 2021 ---
Successfully excluded 1,945 transaction records from the year 2021.
Final merged dataset row count (excluding 2021): 2,049,263
Consolidated Single Source of Truth successfully exported to: '..\data\processed\master_df.csv'
